In [1]:
import os

from inlist import target, PIPELINE_OPTIONS_OVERRIDE
from run_context import apply_context_globals
from atmos_lbl_pipeline import step_log
from abund_lbl_pipeline import (
    load_star_and_atmosphere_context,
    plot_all_element_gaussian_fits,
    run_nonfe_line_review,
    apply_lineqa_for_abundance,
    prepare_abundance_engine,
    compute_lbl_differential_and_export,
    run_hfs_blends_and_update_outputs,
)

ctx = apply_context_globals(
    globals(),
    target=target,
    pipeline_options_override=PIPELINE_OPTIONS_OVERRIDE,
    prepare_io=True,
)

- In previous steps, atmospheric parameters for this spectrum (Teff, logg, [Fe/H], vmic) were already computed and stored in the `_q2.dump` file associated with `output_atmos_params_dumpfile_path`.

- This notebook keeps those parameters fixed, then computes elemental abundances using EW + `ispec.determine_abundances` (`code="moog"`), and performs line-by-line differential analysis against the solar per-line library.

- The key input is `*_melendez2014_star_fitted_linemasks.txt`, which must include cross-match and fitting metadata (optionally filtered by LineQA).


In [2]:
step_log("1", "Load context inputs for element abundance workflow.")
_ctx4a = load_star_and_atmosphere_context(
    ispec=ispec,
    target=target,
    output_folder=output_folder,
    spectrum_norm_path=spectrum_norm_path,
    output_atmos_params_dumpfile_path=output_atmos_params_dumpfile_path,
    initial_teff=initial_teff,
    initial_logg=initial_logg,
    initial_MH=initial_MH,
)

star_spectrum = _ctx4a["star_spectrum"]
linemask_output_folder = _ctx4a["linemask_output_folder"]
linemasks = _ctx4a["linemasks"]
status_q2 = _ctx4a["status_q2"]
teff = _ctx4a["teff"]
logg = _ctx4a["logg"]
mh = _ctx4a["mh"]
alpha = _ctx4a["alpha"]
vmic = _ctx4a["vmic"]

[STEP 1] Load context inputs for element abundance workflow.
[STEP 1] Loading spectrum, fitted linemasks, and atmospheric dump.
[STEP 1] Loaded fitted linemasks: 227
[STEP 1] Reference parameters: Teff=5720.0, logg=4.46, [Fe/H]=-0.09
[STEP 1] Fitted parameters: Teff=5721.00±3.00, logg=4.49±0.01, [M/H]=-0.08±0.00, vmic=0.97±0.01


In [3]:
step_log("2", "Run Gaussian diagnostics for all non-Fe elements.")
RUN_FIND_LINEMASKS = PIPELINE_OPTIONS.get("run_find_linemasks", True)
USE_EXTERNAL_EW = PIPELINE_OPTIONS.get("input_has_external_ews", False)
RUN_PART1_SPECTRUM = RUN_FIND_LINEMASKS and (not USE_EXTERNAL_EW)
PLOT_All_Elements = True

_plot_res = plot_all_element_gaussian_fits(
    star_spectrum=star_spectrum,
    linemasks=linemasks,
    output_folder=output_folder,
    run_part1_spectrum=RUN_PART1_SPECTRUM,
    plot_all_elements=PLOT_All_Elements,
    progress_every=10,
)
fig_ele_dir = _plot_res["base_output_dir"]
step_log("2", f"Element Gaussian diagnostics ready: dir={fig_ele_dir}")

[STEP 2] Run Gaussian diagnostics for all non-Fe elements.
[STEP 2] Plotting Gaussian diagnostics for all non-Fe elements.
[STEP 2][0/27] Element plotting started.
[STEP 2][1/27] Element Al 1: rendered=2/2
[STEP 2][2/27] Element Ba 2: rendered=2/2
[STEP 2][3/27] Element C 1: rendered=1/1
[STEP 2][4/27] Element Ca 1: rendered=10/10
[STEP 2][5/27] Element Ce 2: rendered=7/7
[STEP 2][6/27] Element Co 1: rendered=8/8
[STEP 2][7/27] Element Cr 1: rendered=9/9
[STEP 2][8/27] Element Cr 2: rendered=5/5
[STEP 2][9/27] Element Cu 1: rendered=3/3
[STEP 2][10/27] Element Eu 2: rendered=2/2
[STEP 2][11/27] Element Mg 1: rendered=2/2
[STEP 2][12/27] Element Mn 1: rendered=2/2
[STEP 2][13/27] Element Na 1: rendered=4/4
[STEP 2][14/27] Element Nd 2: rendered=4/4
[STEP 2][15/27] Element Ni 1: rendered=10/17
[STEP 2][15/27] Element Ni 1: rendered=17/17
[STEP 2][16/27] Element S 1: rendered=2/2
[STEP 2][17/27] Element Sc 1: rendered=1/1
[STEP 2][18/27] Element Sc 2: rendered=3/3
[STEP 2][19/27] Element 

In [4]:
step_log("3", "Run interactive line review for non-Fe Gaussian plots.")
run_nonfe_line_review(
    output_folder=output_folder,
    target=target,
    run_part1_spectrum=RUN_PART1_SPECTRUM,
    plot_all_elements=PLOT_All_Elements,
    pipeline_options=PIPELINE_OPTIONS,
)

[STEP 3] Run interactive line review for non-Fe Gaussian plots.
[STEP 3] Starting interactive line review for non-Fe lines.
[LineReview] target=HIP1954 | images=150
[LineReview] opening UI in mode='browser' -> http://127.0.0.1:60193/
[LineReview] Saved. 47 deleted rows updated in Lines_deleted.tsv; 47 images currently in deleted/; summary=47.


In [5]:
step_log("4", "Apply LineQA updates to linemasks for abundance computation.")
linemasks = apply_lineqa_for_abundance(
    ispec=ispec,
    linemasks=linemasks,
    output_folder=output_folder,
    linemask_output_folder=linemask_output_folder,
    target=target,
    pipeline_options=PIPELINE_OPTIONS,
)

[STEP 4] Apply LineQA updates to linemasks for abundance computation.
[STEP 4] Applying LineQA (deleted lines + EW modifications).
[STEP 4] LineQA applied: kept=179, deleted=61, modified=0
[STEP 4] Saved filtered linemasks for abundance: /Users/jiayue/iSpec/output/MegantestSample/HIP14614/linemasks/HIP14614_melendez2014_star_fitted_linemasks.filtered_for_abund.txt


## Start abundance calculation

- Refer to `determine_abundances_from_ew` in `example.py`.

- Atmospheric model: `model = ispec_dir + "/input/atmospheres/MARCS.GES/"` (small grid with preliminary interpolation).

- Solar abundance model: `ispec_dir + "/input/abundances/Grevesse.2007/stdatom.dat"

In [6]:
step_log("5", "Prepare abundance engine (model atmosphere + solar references).")
linemasks = linemasks[linemasks["wave_nm"] > 0]
linemasks = linemasks[linemasks["ew"] > 0]

_engine = prepare_abundance_engine(
    ispec=ispec,
    ispec_dir=ispec_dir,
    teff=teff,
    logg=logg,
    mh=mh,
    alpha=alpha,
    code="moog",
)
code = _engine["code"]
model = _engine["model"]
solar_abundances_file = _engine["solar_abundances_file"]
solar_abundances = _engine["solar_abundances"]
atmosphere_layers = _engine["atmosphere_layers"]
step_log("5", f"Abundance engine ready: model={model}")

[STEP 5] Prepare abundance engine (model atmosphere + solar references).
[STEP 5] Preparing atmosphere model and solar abundance reference.
[STEP 5] Abundance engine ready: model=/Users/jiayue/iSpec/input/atmospheres/MARCS.GES/


## Line-by-line differential analysis

In [7]:

step_log("6", "Compute LBL differential abundances and export outputs.")
_lbl = compute_lbl_differential_and_export(
    ispec=ispec,
    linemasks=linemasks,
    target=target,
    ispec_dir=ispec_dir,
    instrument_name=instrument_name,
    from_resolution=from_resolution,
    solar_lines_csv=globals().get("solar_lines_csv"),
    atmosphere_layers=atmosphere_layers,
    solar_abundances=solar_abundances,
    teff=teff,
    logg=logg,
    mh=mh,
    alpha=alpha,
    vmic=vmic,
    code=code,
    output_abundances_result_path_linebyline_q2=output_abundances_result_path_linebyline_q2,
)
raw_path = _lbl["raw_path"]
print(_lbl["summary_df"].head(10))

'''
def norm_elem(e):
    s = str(e).strip()
    s = s.replace(" 1", " I").replace(" 2", " II")
    return s

# ---------- 1) 读取 solar_lines_instru（你 0_ 写出来的逐线太阳库） ----------
solar_lines_csv = globals().get("solar_lines_csv") or os.path.join(
    ispec_dir, "input", "solar_lines_instru.csv"
)
if not os.path.isfile(solar_lines_csv):
    raise FileNotFoundError(f"找不到太阳逐线库 CSV: {solar_lines_csv}")
sun_master = pd.read_csv(solar_lines_csv)

# ---------- NEW: instrument 先筛出来 ----------
sun_inst = sun_master[sun_master["instrument"].astype(str).str.strip() == str(instrument_name).strip()].copy()
if len(sun_inst) == 0:
    raise RuntimeError(f"[export_q2_inputs] solar master has no rows for instrument={instrument_name}")

# ---------- NEW: 自动选最接近的 resolution ----------
# 注意：这里转 float 是为了兼容 int/float/字符串等情况
avail = pd.to_numeric(sun_inst["resolution"], errors="coerce").values
if not np.isfinite(avail).any():
    raise RuntimeError(f"[export_q2_inputs] solar master resolution column cannot be parsed for instrument={instrument_name}")

req = float(from_resolution)
res_use = float(avail[np.nanargmin(np.abs(avail - req))])
res_series = pd.to_numeric(sun_inst["resolution"], errors="coerce")
sun_sel = sun_inst[np.isclose(res_series, res_use, rtol=0, atol=1e-3)].copy()
if sun_sel.empty:
    sun_sel = sun_inst.copy()
    print(
        f"[warn] solar_lines_instru: no rows at R={res_use:g} for {instrument_name}; "
        f"using all {len(sun_inst)} instrument rows."
    )
else:
    print(f"solar_lines_instru: using R={res_use:g} ({len(sun_sel)} rows)")

sun_sel["element_norm"] = sun_sel["element"].map(norm_elem)

# ---------- 2) 准备 iSpec abundance 所需：solar abund file + model atmos ----------
# 这些变量你在 userlist.py / 前面 cell 通常已经有：
# model, code, solar_abundances_file
solar_abundances = ispec.read_solar_abundances(solar_abundances_file)
modeled_layers_pack = ispec.load_modeled_layers_pack(model)

# 用 q2 的 teff/logg/mh/alpha 生成 atmosphere_layers
atmosphere_layers = ispec.interpolate_atmosphere_layers(
    modeled_layers_pack,
    {"teff": teff, "logg": logg, "MH": mh, "alpha": alpha},
    code=code
)

microturbulence_vel = vmic

# ---------- 3) 对 star 的每个元素逐线算 logeps & [X/H] ----------
elem_norm_all = np.array([norm_elem(e) for e in linemasks["element"]])

# 如果你有 discarded 字段，先过滤掉（强烈建议）
if "discarded" in linemasks.dtype.names:
    disc = linemasks["discarded"]

    # 1) 如果本来就是 bool
    if disc.dtype == np.bool_:
        mask_use = ~disc

    # 2) 数值型：0 表示未丢弃，非0 表示丢弃（按你的数据习惯也合理）
    elif np.issubdtype(disc.dtype, np.number):
        mask_use = (disc == 0)

    # 3) 字符串/对象型：识别 "True/False", "1/0", "yes/no"
    else:
        disc_str = np.array([str(x).strip().lower() for x in disc])
        is_discarded = np.isin(disc_str, ["true", "1", "yes", "y", "t"])
        mask_use = ~is_discarded
else:
    mask_use = np.ones(len(linemasks), dtype=bool)

linemasks_use = linemasks[mask_use]
elem_norm_use = elem_norm_all[mask_use]

elements = np.unique(elem_norm_use)

star_lines = []
for ele in elements:
    sel = (elem_norm_use == ele)
    lm_ele = linemasks_use[sel]
    if len(lm_ele) == 0:
        continue

    spec_abund, normal_abund, x_over_h, x_over_fe = ispec.determine_abundances(
        atmosphere_layers,
        teff, logg, mh, alpha,
        lm_ele,
        solar_abundances,
        microturbulence_vel=microturbulence_vel,
        verbose=0,
        code=code
    )

    spec_abund = np.asarray(spec_abund, dtype=float)  # logeps-12
    x_over_h   = np.asarray(x_over_h, dtype=float)

    for lm, logeps_m12, xh in zip(lm_ele, spec_abund, x_over_h):
        if not (np.isfinite(logeps_m12) and np.isfinite(xh)):
            continue
        star_lines.append({
            "id": str(target),
            "element": ele,
            "wave_A": float(lm["wave_A"]),
            "EP": float(lm["lower_state_eV"]),
            "loggf": float(lm["loggf"]),
            "ew_mA": float(lm["ew"]),
            "logeps_star": float(logeps_m12 + 12.0),
            "[X/H]_star_Grevesse": float(xh),
        })

df_star_lines = pd.DataFrame(star_lines)
print("star per-line rows:", len(df_star_lines))

# ---------- 4) 与太阳逐线库 inner 对齐（波长键 0.001 Å） ----------
_wave_tol_A = 0.001

def _wave_keys_A(wa):
    w = np.asarray(wa, dtype=float)
    return np.round(w / _wave_tol_A).astype(np.int64) * _wave_tol_A

sun_m = sun_sel[["element_norm", "wave_A", "logeps_sun", "[X/H]_sun_Grevesse"]].copy()
sun_m = sun_m.rename(columns={"element_norm": "element"})
sun_m["wave_key"] = _wave_keys_A(sun_m["wave_A"])
sun_m = sun_m.drop_duplicates(subset=["element", "wave_key"], keep="first")

df_s = df_star_lines.copy()
df_s["wave_key"] = _wave_keys_A(df_s["wave_A"])

df_merge = df_s.merge(
    sun_m[["element", "wave_key", "logeps_sun", "[X/H]_sun_Grevesse"]],
    on=["element", "wave_key"],
    how="inner",
)

# 逐线差分（两种都给你：Δlogeps 和 Δ[X/H]）
df_merge["dlogeps"] = df_merge["logeps_star"] - df_merge["logeps_sun"]
df_merge["d[X/H]"]  = df_merge["[X/H]_star_Grevesse"] - df_merge["[X/H]_sun_Grevesse"]

# 保存 line-by-line
df_merge_out = df_merge[[
    "id","element","wave_A","EP","loggf","ew_mA",
    "logeps_star","logeps_sun","dlogeps",
    "[X/H]_star_Grevesse","[X/H]_sun_Grevesse","d[X/H]"
]].copy()

#df_merge_out.to_csv(output_abundances_result_path_linebyline_q2, index=False)
#print("Saved line-by-line:", output_abundances_result_path_linebyline_q2)
# ===================== 4_ 输出两份文件 =====================

# ---- (A) 逐线表：保留给你以后检查/debug ----
raw_path = output_abundances_result_path_linebyline_q2.replace(".csv", "_raw.csv")
df_merge_out.to_csv(raw_path, index=False)
print("Saved raw per-line:", raw_path)

# ---- (B) 给 5_ 用的汇总表：按 element×ion 汇总成 5_ 需要的格式 ----
ROMAN2INT = {"I":1, "II":2, "III":3, "IV":4}

def to_num_ion_token(s):
    """把 'Fe I' -> 'Fe 1'；'Ti II' -> 'Ti 2'；'C' -> 'C 1' """
    parts = str(s).strip().split()
    if len(parts) == 1:
        return f"{parts[0]} 1"
    base, ion = parts[0], parts[1]
    ion_u = ion.upper()
    if ion_u in ROMAN2INT:
        return f"{base} {ROMAN2INT[ion_u]}"
    # 万一本来就是数字
    try:
        return f"{base} {int(float(ion))}"
    except:
        return f"{base} 1"

df_for_plot = df_merge_out.copy()
df_for_plot["element"] = df_for_plot["element"].apply(to_num_ion_token)

# 这里用 d[X/H] 作为 line-by-line differential 的 [X/H]
# std_[X/H] 用线间散布（ddof=1），n_lines 用计数
g = df_for_plot.groupby("element")["d[X/H]"]
df_summary = g.agg(
    n_lines="count",
    **{"[X/H]":"mean"}
).reset_index()

# 手动算 std（避免只有 1 条线时 std=nan 影响画图，你也可以保留 nan）
std = g.std(ddof=1).reset_index(drop=True)
df_summary["std_[X/H]"] = std

# 排序：Fe 1, Fe 2 放前面（和你原来习惯一致）
order = []
for x in ["Fe 1", "Fe 2"]:
    if x in df_summary["element"].values:
        order.append(x)
others = [x for x in df_summary["element"].tolist() if x not in order]
df_summary["__ord"] = pd.Categorical(df_summary["element"], categories=order+sorted(others), ordered=True)
df_summary = df_summary.sort_values("__ord").drop(columns="__ord").reset_index(drop=True)

# 覆盖写回：让 5_ 直接读这个文件（不需要改 5_）
df_summary.to_csv(output_abundances_result_path_linebyline_q2, index=False)
print("Saved summary for 5_:", output_abundances_result_path_linebyline_q2)

'''

[STEP 6] Compute LBL differential abundances and export outputs.
[STEP 6] Computing line-by-line abundances and differential [X/H].
[STEP 6] Using solar line library at R=115000, rows=194
[STEP 6] Element abundance progress: 5/25
[STEP 6] Element abundance progress: 10/25
[STEP 6] Element abundance progress: 15/25
[STEP 6] Element abundance progress: 20/25
[STEP 6] Element abundance progress: 25/25
[STEP 6] Computed star per-line rows: 179
[STEP 6] Saved raw per-line abundance file: /Users/jiayue/iSpec/output/MegantestSample/HIP14614/abundances_linebyline_q2_HIP14614_raw.csv
[STEP 6] Saved summary abundance file for Step 5 plotting: /Users/jiayue/iSpec/output/MegantestSample/HIP14614/abundances_linebyline_q2_HIP14614.csv
  element  n_lines     [X/H]  std_[X/H]
0    Fe 1       65 -0.098092   0.020888
1    Fe 2        9 -0.104111   0.011374
2    Al 1        2 -0.110500   0.000707
3    Ba 2        2 -0.030000   0.026870
4     C 1        1 -0.122000        NaN
5    Ca 1        9 -0.080333 

'\ndef norm_elem(e):\n    s = str(e).strip()\n    s = s.replace(" 1", " I").replace(" 2", " II")\n    return s\n\n# ---------- 1) 读取 solar_lines_instru（你 0_ 写出来的逐线太阳库） ----------\nsolar_lines_csv = globals().get("solar_lines_csv") or os.path.join(\n    ispec_dir, "input", "solar_lines_instru.csv"\n)\nif not os.path.isfile(solar_lines_csv):\n    raise FileNotFoundError(f"找不到太阳逐线库 CSV: {solar_lines_csv}")\nsun_master = pd.read_csv(solar_lines_csv)\n\n# ---------- NEW: instrument 先筛出来 ----------\nsun_inst = sun_master[sun_master["instrument"].astype(str).str.strip() == str(instrument_name).strip()].copy()\nif len(sun_inst) == 0:\n    raise RuntimeError(f"[export_q2_inputs] solar master has no rows for instrument={instrument_name}")\n\n# ---------- NEW: 自动选最接近的 resolution ----------\n# 注意：这里转 float 是为了兼容 int/float/字符串等情况\navail = pd.to_numeric(sun_inst["resolution"], errors="coerce").values\nif not np.isfinite(avail).any():\n    raise RuntimeError(f"[export_q2_inputs] solar master resoluti

In [8]:
step_log("7", "Run HFS processing and update line-by-line outputs.")
RUN_HFS_BLENDS = True
RUN_HFS_BLENDS_DRY_RUN = False

_hfs = run_hfs_blends_and_update_outputs(
    ispec=ispec,
    target=target,
    output_folder=output_folder,
    ispec_dir=ispec_dir,
    atmosphere_layers=atmosphere_layers,
    teff=teff,
    logg=logg,
    mh=mh,
    output_abundances_result_path_linebyline_q2=output_abundances_result_path_linebyline_q2,
    pipeline_options=PIPELINE_OPTIONS,
    run_hfs_blends=RUN_HFS_BLENDS,
    dry_run=RUN_HFS_BLENDS_DRY_RUN,
)
if _hfs.get("ran"):
    step_log(
        "7",
        f"HFS done: rows={_hfs['hfs_rows']}, replaced={_hfs['replaced_n']}, failed={_hfs['failed_hfs_n']}",
    )
    if _hfs.get("report_path"):
        print(f"HFS detailed report saved: {_hfs['report_path']}")

'''
# HFS 后处理：
# 1) 选 HFS 元素 (Cu/Mn/Co/Ba/V)
# 2) 读取 q2 line-by-line 对应线和 EW
# 3) linemake 生成该线的 HFS 分量
# 4) MOOG blends 计算 abundance
# 5) 保存 CSV 供后续绘图

RUN_HFS_BLENDS = True
RUN_HFS_BLENDS_DRY_RUN = False

PIPELINE_OPTIONS = globals().get("PIPELINE_OPTIONS", {})
# input_has_external_ews=True：仍跑完整 HFS；锚定 EW 以 q2_work/lines_q2.csv 目标列为输入（见 run_hfs_blends_for_target(..., use_external_input_ew=True)）
USE_EXTERNAL_INPUT_EW_FOR_HFS = bool(PIPELINE_OPTIONS.get("input_has_external_ews", False))
if USE_EXTERNAL_INPUT_EW_FOR_HFS:
    print(
        "[4a] input_has_external_ews=True：HFS 使用 q2_work/lines_q2.csv 中该目标的 EW 列作为 MOOG blends 锚定（已有锚定时不再用 raw 的 ew_mA 覆盖）"
    )

if RUN_HFS_BLENDS:
    hfs_model_in = os.path.join(output_folder, "HFS", "runs", f"model_{target}.in")
    os.makedirs(os.path.dirname(hfs_model_in), exist_ok=True)

    ispec.write_atmosphere(
        atmosphere_layers,
        teff,
        logg,
        MH,
        atmosphere_filename=hfs_model_in,
        code="moog",
    )

    if (not os.path.exists(hfs_model_in)) or os.path.getsize(hfs_model_in) == 0:
        raise RuntimeError(f"Failed to export MOOG model atmosphere: {hfs_model_in}")
    print(f"HFS model exported: {hfs_model_in}")

    linemake_bin_cfg = os.environ.get("ISPEC_LINEMAKE_BIN") or os.environ.get("LINEMAKE_BIN")
    if not linemake_bin_cfg:
        _local_linemake = os.path.join(ispec_dir, "tools", "linemake", "linemake.go")
        if os.path.exists(_local_linemake):
            linemake_bin_cfg = _local_linemake
            os.environ["ISPEC_LINEMAKE_BIN"] = linemake_bin_cfg

    print(f"linemake_bin config: {linemake_bin_cfg}")

    hfs_df = run_hfs_blends_for_target(
        target=target,
        output_folder=output_folder,
        ispec_dir=ispec_dir,
        model_in=hfs_model_in,
        ew_result_csv=output_abundances_result_path_linebyline_q2,
        moog_bin=None,
        linemake_bin=linemake_bin_cfg,
        window_A=2.0,
        dry_run=RUN_HFS_BLENDS_DRY_RUN,
        require_linemake=True,
        enable_implausible_hfs_shift_gate=True,
        use_external_input_ew=USE_EXTERNAL_INPUT_EW_FOR_HFS,
    )

    hfs_out = os.path.join(output_folder, "HFS", "tables", f"abundances_hfs_blends_{target}.csv")
    hfs_sum_out = os.path.join(output_folder, "HFS", "tables", f"abundances_hfs_blends_{target}_summary.csv")
    print(f"HFS rows: {len(hfs_df)}  ->  {hfs_out}")

    if len(hfs_df):
        print(hfs_df[["element", "wave_A", "n_hfs_components", "status"]].head(10))
        ok_n = int((hfs_df["status"].astype(str) == "ok").sum())
        print(f"HFS ok rows: {ok_n}/{len(hfs_df)}")

    if os.path.exists(hfs_sum_out):
        hfs_sum = pd.read_csv(hfs_sum_out)
        show_cols = [c for c in ["element", "n_lines_total", "n_lines", "n_ok", "xh_mean", "xfe_mean"] if c in hfs_sum.columns]
        if show_cols:
            print("HFS summary:")
            print(hfs_sum[show_cols])
    else:
        hfs_sum = pd.DataFrame()

    # ---- 将 HFS 结果回写到 q2 raw/summary，并新增 hfs 标记列 ----
    raw_path = output_abundances_result_path_linebyline_q2.replace(".csv", "_raw.csv")
    sum_path = output_abundances_result_path_linebyline_q2
    HFS_ELEMS = {"Cu", "Mn", "Co", "Ba", "V"}

    def _canon_ele_ion(ele_text):
        s = str(ele_text).strip().replace("  ", " ")
        parts = s.split(" ")
        sym = parts[0] if parts else s
        ion_raw = parts[1] if len(parts) > 1 else "1"
        ion_u = str(ion_raw).upper()
        if ion_u in ["I", "1"]:
            ion = "1"
        elif ion_u in ["II", "2"]:
            ion = "2"
        else:
            ion = str(ion_raw)
        return sym, ion

    # 逐线 raw：
    # - 非 HFS 元素: hfs=""
    # - HFS 元素且成功替换: hfs=替换前 no-HFS 的 d[X/H]
    # - HFS 元素但未成功替换: hfs="f"
    if os.path.exists(raw_path):
        raw_df = pd.read_csv(raw_path)
        raw_df["wave_A"] = pd.to_numeric(raw_df.get("wave_A"), errors="coerce")

        hfs_ok = hfs_df.copy()
        hfs_ok = hfs_ok[
            (hfs_ok.get("mode", "hfs").astype(str) == "hfs")
            & (hfs_ok["status"].astype(str) == "ok")
        ].copy()
        hfs_ok["wave_A"] = pd.to_numeric(hfs_ok.get("wave_A"), errors="coerce")
        hfs_ok["logeps_HFS"] = pd.to_numeric(hfs_ok.get("logeps_HFS"), errors="coerce")
        hfs_ok["[X/H]_HFS"] = pd.to_numeric(hfs_ok.get("[X/H]_HFS"), errors="coerce")
        hfs_ok = hfs_ok[np.isfinite(hfs_ok["wave_A"]) & np.isfinite(hfs_ok["logeps_HFS"]) & np.isfinite(hfs_ok["[X/H]_HFS"])].copy()

        raw_df["hfs"] = ""
        replaced_n = 0
        failed_hfs_n = 0

        for i, r in raw_df.iterrows():
            ele_raw = str(r.get("element", "")).strip()
            sym, ion = _canon_ele_ion(ele_raw)
            if sym not in HFS_ELEMS:
                raw_df.at[i, "hfs"] = ""
                continue

            w = pd.to_numeric(r.get("wave_A", np.nan), errors="coerce")
            old_dxh = pd.to_numeric(r.get("d[X/H]", np.nan), errors="coerce")

            cand = hfs_ok[
                (hfs_ok["element"].astype(str).str.strip() == sym)
                & (hfs_ok["ion"].astype(str).str.strip() == ion)
            ].copy()
            if np.isfinite(w) and len(cand):
                cand["dw"] = (cand["wave_A"] - float(w)).abs()
                cand = cand.sort_values("dw")
                if float(cand.iloc[0]["dw"]) <= 0.003:
                    b = cand.iloc[0]
                    raw_df.at[i, "hfs"] = old_dxh if np.isfinite(old_dxh) else ""
                    raw_df.at[i, "logeps_star"] = float(b["logeps_HFS"])
                    raw_df.at[i, "d[X/H]"] = float(b["[X/H]_HFS"])
                    if "[X/H]_sun_Grevesse" in raw_df.columns:
                        xh_sun = pd.to_numeric(raw_df.at[i, "[X/H]_sun_Grevesse"], errors="coerce")
                        if np.isfinite(xh_sun):
                            raw_df.at[i, "[X/H]_star_Grevesse"] = float(b["[X/H]_HFS"]) + float(xh_sun)
                        else:
                            raw_df.at[i, "[X/H]_star_Grevesse"] = float(b["[X/H]_HFS"])
                    if "logeps_sun" in raw_df.columns:
                        lsun = pd.to_numeric(raw_df.at[i, "logeps_sun"], errors="coerce")
                        if np.isfinite(lsun):
                            raw_df.at[i, "dlogeps"] = float(b["logeps_HFS"]) - float(lsun)
                    replaced_n += 1
                    continue

            raw_df.at[i, "hfs"] = "f"
            failed_hfs_n += 1

        raw_df.to_csv(raw_path, index=False)
        print(f"raw updated with HFS: replaced={replaced_n}, failed={failed_hfs_n} -> {raw_path}")

    # 元素汇总 summary：
    # - 非 HFS 元素: hfs="n"
    # - HFS 元素且存在替换: hfs="y"
    # - HFS 元素但 n_ok=0/无可替换结果: hfs="f"
    if os.path.exists(sum_path):
        sum_df = pd.read_csv(sum_path)
        sum_df["hfs"] = "n"

        # 兼容 element 写法（Fe 1 / Fe I）
        def _base_sym(e):
            return _canon_ele_ion(e)[0]

        if len(hfs_sum):
            hs = hfs_sum.copy()
            hs["n_ok"] = pd.to_numeric(hs.get("n_ok"), errors="coerce")
            hs["xh_mean"] = pd.to_numeric(hs.get("xh_mean"), errors="coerce")
            hs["xh_std"] = pd.to_numeric(hs.get("xh_std"), errors="coerce")

            for i, r in sum_df.iterrows():
                sym = _base_sym(r.get("element", ""))
                if sym not in HFS_ELEMS:
                    sum_df.at[i, "hfs"] = "n"
                    continue

                m = hs[hs["element"].astype(str).str.strip() == sym].copy()
                if m.empty:
                    sum_df.at[i, "hfs"] = "f"
                    continue

                mr = m.iloc[0]
                n_ok = pd.to_numeric(mr.get("n_ok", np.nan), errors="coerce")
                if np.isfinite(n_ok) and int(n_ok) > 0 and np.isfinite(pd.to_numeric(mr.get("xh_mean", np.nan), errors="coerce")):
                    sum_df.at[i, "[X/H]"] = float(mr["xh_mean"])
                    if "std_[X/H]" in sum_df.columns and np.isfinite(pd.to_numeric(mr.get("xh_std", np.nan), errors="coerce")):
                        sum_df.at[i, "std_[X/H]"] = float(mr["xh_std"])
                    sum_df.at[i, "hfs"] = "y"
                else:
                    sum_df.at[i, "hfs"] = "f"
        else:
            for i, r in sum_df.iterrows():
                sym = _base_sym(r.get("element", ""))
                if sym in HFS_ELEMS:
                    sum_df.at[i, "hfs"] = "f"

        sum_df.to_csv(sum_path, index=False)
        y_n = int((sum_df["hfs"].astype(str) == "y").sum())
        f_n = int((sum_df["hfs"].astype(str) == "f").sum())
        n_n = int((sum_df["hfs"].astype(str) == "n").sum())
        print(f"summary updated with HFS flags: y={y_n}, f={f_n}, n={n_n} -> {sum_path}")
        _ew_flag = " --external-ew" if PIPELINE_OPTIONS.get("input_has_external_ews") else ""
        print(
            "[4a] Bedell 对比图读取 abundances_linebyline_q2_<target>.csv；"
            f"重跑 7_compare 均值图前请执行: python Code/bedell_jiayue_agreement_stats.py{_ew_flag}"
        )

    cmp_df, cmp_fig = build_ew_vs_hfs_diagnostics(
        target=target,
        output_folder=output_folder,
    )
    print(f"EW-vs-HFS comparison rows: {len(cmp_df)}")
    if cmp_fig:
        print("EW-vs-HFS figure:", cmp_fig)

    # ---- HFS 运行报告（逐元素/逐线原因）并保存到 tables ----
    tables_dir = os.path.join(output_folder, "HFS", "tables")
    os.makedirs(tables_dir, exist_ok=True)
    report_path = os.path.join(tables_dir, f"hfs_run_report_{target}.txt")

    target_lines_path = os.path.join(tables_dir, f"hfs_target_lines_{target}.csv")
    linemake_runs_path = os.path.join(tables_dir, f"linemake_runs_{target}.csv")
    manifest_path = os.path.join(tables_dir, f"blends_manifest_{target}.csv")

    rep = hfs_df.copy()
    for col in ["wave_A", "species", "n_hfs_components"]:
        if col in rep.columns:
            rep[col] = pd.to_numeric(rep[col], errors="coerce")

    if os.path.exists(target_lines_path):
        tl = pd.read_csv(target_lines_path)
        for col in ["wave_A", "species", "ep_eV", "d_xh_ref", "logeps_star_ref"]:
            if col in tl.columns:
                tl[col] = pd.to_numeric(tl[col], errors="coerce")
        keep = [c for c in ["element", "wave_A", "species", "ep_eV", "d_xh_ref", "logeps_star_ref"] if c in tl.columns]
        rep = rep.merge(tl[keep], on=["element", "wave_A", "species"], how="left")

    if os.path.exists(linemake_runs_path):
        lm = pd.read_csv(linemake_runs_path)
        for col in ["wave_A", "species", "n_components"]:
            if col in lm.columns:
                lm[col] = pd.to_numeric(lm[col], errors="coerce")
        keep = [c for c in ["element", "wave_A", "species", "n_components"] if c in lm.columns]
        rep = rep.merge(lm[keep], on=["element", "wave_A", "species"], how="left")
        if "n_components" in rep.columns:
            rep = rep.rename(columns={"n_components": "linemake_n_components"})

    if os.path.exists(manifest_path):
        mf = pd.read_csv(manifest_path)
        for col in ["wave_A", "species", "n_hfs_components_blends"]:
            if col in mf.columns:
                mf[col] = pd.to_numeric(mf[col], errors="coerce")
        keep = [c for c in ["element", "wave_A", "species", "n_hfs_components_blends", "component_qc", "component_source"] if c in mf.columns]
        rep = rep.merge(mf[keep], on=["element", "wave_A", "species"], how="left")

    def _reason_of_row(r):
        st = str(r.get("status", ""))
        note = str(r.get("note", ""))
        ncomp = pd.to_numeric(r.get("n_hfs_components", np.nan), errors="coerce")
        lmcomp = pd.to_numeric(r.get("linemake_n_components", np.nan), errors="coerce")
        if st == "ok":
            return "ok"
        if st == "invalid_no_hfs_component":
            if np.isfinite(lmcomp):
                return f"linemake可用分量不足: n_components={int(lmcomp)} (<2)"
            if np.isfinite(ncomp):
                return f"HFS分量不足: n_hfs_components={int(ncomp)} (<2)"
            return "HFS分量不足(<2)"
        if "no_anchor_line_match" in note:
            return "blends输出中无锚线匹配(波长/跃迁不对应)"
        if "parsed_element_line_not_allowed" in note:
            return "blends仅给出element级abundance，缺少species锚线块"
        if "implausible_hfs_shift" in note:
            return "相对q2同线偏差过大，被判定为不可信解"
        if "no_valid_abundance_rows_in_species_block" in note:
            return "species块存在，但有效abundance行不可用"
        return f"未分类: {st} | {note}"

    rep["reason"] = rep.apply(_reason_of_row, axis=1)

    lines = []
    lines.append(f"HFS Run Report for {target}")
    lines.append("=" * 72)
    lines.append(f"model_in: {hfs_model_in}")
    lines.append(f"linemake_bin: {linemake_bin_cfg}")
    lines.append(f"hfs_rows: {len(rep)}")
    if len(rep):
        lines.append(f"hfs_ok_rows: {int((rep['status'].astype(str) == 'ok').sum())}/{len(rep)}")
    lines.append("")

    for elem in ["Cu", "Mn", "Co", "Ba", "V"]:
        d = rep[rep["element"].astype(str) == elem].copy().sort_values("wave_A")
        if d.empty:
            continue
        n_ok = int((d["status"].astype(str) == "ok").sum())
        lines.append(f"[{elem}] n_ok={n_ok}/{len(d)}")
        for _, rr in d.iterrows():
            w = pd.to_numeric(rr.get("wave_A", np.nan), errors="coerce")
            ep = pd.to_numeric(rr.get("ep_eV", np.nan), errors="coerce")
            ncomp = pd.to_numeric(rr.get("n_hfs_components", np.nan), errors="coerce")
            lmcomp = pd.to_numeric(rr.get("linemake_n_components", np.nan), errors="coerce")
            q2x = pd.to_numeric(rr.get("d_xh_ref", np.nan), errors="coerce")
            hx = pd.to_numeric(rr.get("[X/H]_HFS", np.nan), errors="coerce")
            st = str(rr.get("status", ""))
            rs = str(rr.get("reason", ""))
            q2s = f"{q2x:.3f}" if np.isfinite(q2x) else "nan"
            hxs = f"{hx:.3f}" if np.isfinite(hx) else "nan"
            src = str(rr.get("hfs_source_used", rr.get("component_source", "unknown")))
            lines.append(
                "  - line "
                f"{w:.3f}A | EP={ep:.3f} | status={st} | "
                f"hfs_source={src} | "
                f"n_hfs={int(ncomp) if np.isfinite(ncomp) else 'nan'} | "
                f"linemake_n={int(lmcomp) if np.isfinite(lmcomp) else 'nan'} | "
                f"q2_[X/H]={q2s}"
            )
            lines.append(f"    HFS_[X/H]={hxs}")
            lines.append(f"    reason: {rs}")
        lines.append("")

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

    print(f"HFS run report saved: {report_path}")
    print("\n".join(lines[:20]))

else:
    print("[4a] RUN_HFS_BLENDS=False：跳过 HFS blends（不调 run_hfs_blends_for_target / 不更新 raw、summary）")

'''


[STEP 7] Run HFS processing and update line-by-line outputs.
[STEP 7] Running HFS blends post-processing.
[STEP 7] HFS completed: rows=10, replaced=9, failed=1, ew_vs_hfs_rows=5
[STEP 7] HFS detailed report path: /Users/jiayue/iSpec/output/MegantestSample/HIP14614/HFS/tables/hfs_run_report_HIP14614.txt
[STEP 7] EW-vs-HFS figure: /Users/jiayue/iSpec/output/MegantestSample/HIP14614/HFS/figs/ew_vs_hfs_HIP14614.png
[STEP 7] HFS done: rows=10, replaced=9, failed=1
HFS detailed report saved: /Users/jiayue/iSpec/output/MegantestSample/HIP14614/HFS/tables/hfs_run_report_HIP14614.txt


'\n# HFS 后处理：\n# 1) 选 HFS 元素 (Cu/Mn/Co/Ba/V)\n# 2) 读取 q2 line-by-line 对应线和 EW\n# 3) linemake 生成该线的 HFS 分量\n# 4) MOOG blends 计算 abundance\n# 5) 保存 CSV 供后续绘图\n\nRUN_HFS_BLENDS = True\nRUN_HFS_BLENDS_DRY_RUN = False\n\nPIPELINE_OPTIONS = globals().get("PIPELINE_OPTIONS", {})\n# input_has_external_ews=True：仍跑完整 HFS；锚定 EW 以 q2_work/lines_q2.csv 目标列为输入（见 run_hfs_blends_for_target(..., use_external_input_ew=True)）\nUSE_EXTERNAL_INPUT_EW_FOR_HFS = bool(PIPELINE_OPTIONS.get("input_has_external_ews", False))\nif USE_EXTERNAL_INPUT_EW_FOR_HFS:\n    print(\n        "[4a] input_has_external_ews=True：HFS 使用 q2_work/lines_q2.csv 中该目标的 EW 列作为 MOOG blends 锚定（已有锚定时不再用 raw 的 ew_mA 覆盖）"\n    )\n\nif RUN_HFS_BLENDS:\n    hfs_model_in = os.path.join(output_folder, "HFS", "runs", f"model_{target}.in")\n    os.makedirs(os.path.dirname(hfs_model_in), exist_ok=True)\n\n    ispec.write_atmosphere(\n        atmosphere_layers,\n        teff,\n        logg,\n        MH,\n        atmosphere_filename=hfs_model_in,\n 